# 🤖 RAG-Based Customer FAQ Chatbot — Groq Edition
### University Project — COLAB + Gradio Application

**Architecture:** Document Ingestion → FAISS Semantic Retrieval → Groq LLaMA 3.3 70B → Grounded Answer

---
### ✅ Bugs Fixed in This Version:
- ❌ `null` Python bug in `generate()` function → ✅ Fixed to `None`
- ❌ Gemini 404 API errors → ✅ Replaced with **Groq** (free, fast, no quota issues)
- ❌ `gradio==4.44.1` conflict → ✅ Clean install with latest compatible versions
- ❌ CSS warning in Gradio 6 → ✅ Moved CSS to `launch()`

**Get FREE Groq API key at → [console.groq.com](https://console.groq.com)**

## Step 1 — Install Dependencies

In [1]:
# ── Clean install — no version conflicts ──────────────────────────────────────
!pip install -q groq
!pip install -q gradio
!pip install -q langchain langchain-community langchain-text-splitters
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q pypdf python-docx docx2txt

print('✅ All packages installed!')
print()
print('⚠️  REQUIRED: Runtime → Restart Runtime')
print('⚠️  Then run Steps 2 → 8 in order.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 26.6 MB/s eta 0:00:00
✅ All packages installed!

⚠️  REQUIRED: Runtime → Restart Runtime
⚠️  Then run Steps 2 → 8 in o

## Step 2 — Configure Groq API Key
> Get your **FREE** key at: https://console.groq.com  
> No credit card needed. Very generous free tier!

In [2]:
import os
from getpass import getpass

# ── Paste your Groq key here ──────────────────────────────────────────────────
GROQ_API_KEY = ""
# ─────────────────────────────────────────────────────────────────────────────

if not GROQ_API_KEY:
    GROQ_API_KEY = getpass('🔑 Paste your Groq API key: ')

os.environ['GROQ_API_KEY'] = GROQ_API_KEY

# ── Validate ──────────────────────────────────────────────────────────────────
from groq import Groq
try:
    _client = Groq(api_key=GROQ_API_KEY)
    _resp = _client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[{'role': 'user', 'content': 'Reply with only the word: READY'}],
        max_tokens=10
    )
    print(f'✅ Groq API key valid! Model response: {_resp.choices[0].message.content.strip()}')
    print(f'   Model: llama-3.3-70b-versatile')
except Exception as e:
    print(f'❌ Key error: {e}')
    print('   Get your key at https://console.groq.com')

✅ Groq API key valid! Model response: READY
   Model: llama-3.3-70b-versatile


## Step 3 — Import Libraries

In [3]:
import os
import re
import time
import gradio as gr
from pathlib import Path
from groq import Groq

# LangChain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# Document loaders
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_community.document_loaders import Docx2txtLoader

print('✅ Libraries imported!')

✅ Libraries imported!


## Step 4 — Document Processor

In [4]:
class RAGDocumentProcessor:
    def __init__(self, chunk_size=600, chunk_overlap=100):
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=['\n\n', '\n', '. ', '? ', '! ', ' ', '']
        )

    def load_file(self, path: str) -> list:
        ext = Path(path).suffix.lower()
        try:
            if ext == '.pdf':  return PyPDFLoader(path).load()
            if ext == '.docx': return Docx2txtLoader(path).load()
            if ext == '.txt':  return TextLoader(path, encoding='utf-8').load()
            # fallback
            with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                return [Document(page_content=f.read(), metadata={'source': path})]
        except Exception as e:
            print(f'  ⚠️  Could not load {Path(path).name}: {e}')
            return []

    def load_and_chunk(self, paths: list) -> list:
        docs = []
        for p in paths:
            loaded = self.load_file(p)
            docs.extend(loaded)
            print(f'  📄 {Path(p).name} → {len(loaded)} page(s)')
        chunks = self.splitter.split_documents(docs)
        for i, c in enumerate(chunks):
            c.metadata['chunk_id'] = i
        print(f'  ✂️  {len(chunks)} chunks created')
        return chunks

print('✅ RAGDocumentProcessor ready')

✅ RAGDocumentProcessor ready


## Step 5 — FAISS Vector Store

In [5]:
class RAGVectorStore:
    def __init__(self, model_name='sentence-transformers/all-MiniLM-L6-v2'):
        print('🔄 Loading embedding model...')
        self.emb = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={'device': 'cpu'},
            encode_kwargs={'normalize_embeddings': True}
        )
        self.store = None
        self.count = 0
        print('✅ Embedding model ready')

    def add(self, chunks):
        if not chunks:
            raise ValueError('No chunks provided')
        if self.store is None:
            self.store = FAISS.from_documents(chunks, self.emb)
        else:
            self.store.add_documents(chunks)
        self.count += len(chunks)

    def retrieve(self, query, k=5):
        if not self.store:
            return []
        return self.store.similarity_search_with_score(query, k=k)

    def reset(self):
        self.store = None
        self.count = 0

    @property
    def ready(self):
        return self.store is not None and self.count > 0

print('✅ RAGVectorStore ready')

✅ RAGVectorStore ready


## Step 6 — Groq LLaMA Generator
> **Bug Fixed:** `null` → `None` in `generate()` signature  
> **Provider Changed:** Gemini → **Groq** (llama-3.3-70b-versatile)

In [6]:
class GroqGenerator:
    SYSTEM = """You are a professional, friendly customer support AI.
Answer customer questions using ONLY the company documents provided as context.

Rules:
- Use ONLY the provided context. Never use outside knowledge.
- If the answer is not in the context, say:
  'I don't have that information in our knowledge base. Please contact support directly.'
- Keep answers clear, concise, and helpful.
- Never fabricate prices, dates, phone numbers, or policies."""

    def __init__(self):
        self.client = Groq(api_key=os.environ['GROQ_API_KEY'])
        self.model  = 'llama-3.3-70b-versatile'
        print(f'✅ GroqGenerator ready — model: {self.model}')

    # ── BUG FIX: was `history: list = null`  →  now `history: list = None` ──
    def generate(self, query: str, chunks: list, history: list = None) -> str:
        if not chunks:
            return 'No relevant information found in the uploaded documents.'

        # Build context from retrieved chunks
        ctx_parts = []
        for i, (doc, score) in enumerate(chunks):
            src = Path(doc.metadata.get('source', 'document')).name
            ctx_parts.append(f'[Source {i+1}: {src}]\n{doc.page_content.strip()}')
        context = '\n\n---\n\n'.join(ctx_parts)

        # Build messages list for Groq
        messages = [{'role': 'system', 'content': self.SYSTEM}]

        # Add conversation history (last 4 pairs)
        if history:
            for msg in history[-8:]:   # 8 = 4 pairs
                if isinstance(msg, dict):
                    role    = msg.get('role', 'user')
                    content = msg.get('content', '')
                    # Groq uses 'assistant' not 'model'
                    if role == 'model':
                        role = 'assistant'
                    messages.append({'role': role, 'content': str(content)})

        # Add the current user question with context
        messages.append({
            'role': 'user',
            'content': f'CONTEXT:\n{context}\n\nQUESTION: {query}'
        })

        try:
            resp = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=0.2,
                max_tokens=800,
                top_p=0.85
            )
            return resp.choices[0].message.content.strip()

        except Exception as e:
            err = str(e).lower()
            if '429' in err or 'rate' in err:
                return '⚠️ Rate limit reached. Please wait a moment and try again.'
            return f'❌ Error generating response: {str(e)}'

print('✅ GroqGenerator defined')

✅ GroqGenerator defined


## Step 7 — Initialize RAG System

In [7]:
print('🚀 Initializing RAG system...')

processor     = RAGDocumentProcessor()
vs            = RAGVectorStore()
llm           = GroqGenerator()
indexed_files = []

print('\n✅ All components ready! Run Step 8 to launch the app.')

🚀 Initializing RAG system...
🔄 Loading embedding model...


/tmp/ipykernel_5304/4261804121.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.emb = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model ready
✅ GroqGenerator ready — model: llama-3.3-70b-versatile

✅ All components ready! Run Step 8 to launch the app.


## Step 8 — Gradio Callbacks

In [8]:
def do_upload(files):
    global indexed_files
    if not files:
        return '⚠️  No files selected.'
    new = [f.name for f in files if f.name not in indexed_files]
    if not new:
        return 'ℹ️  All selected files are already indexed.'
    lines = [f'📂 Indexing {len(new)} file(s)...\n']
    try:
        chunks = processor.load_and_chunk(new)
        if not chunks:
            return '❌ No text extracted. Check your files contain readable text.'
        vs.add(chunks)
        indexed_files.extend(new)
        for p in new:
            lines.append(f'  ✅  {Path(p).name}')
        lines.append(f'\n📊 Total chunks in knowledge base: {vs.count}')
    except Exception as e:
        lines.append(f'❌ Error: {e}')
    return '\n'.join(lines)


def do_clear_kb():
    global indexed_files
    vs.reset()
    indexed_files = []
    return '🗑️  Knowledge base cleared.'


def do_chat(user_msg, history):
    history = history or []
    if not user_msg or not user_msg.strip():
        return history, ''

    if not vs.ready:
        bot = '⚠️ **No documents loaded.** Please upload and index files first (Upload Documents tab).'
        history.append({'role': 'user',      'content': user_msg})
        history.append({'role': 'assistant', 'content': bot})
        return history, ''

    try:
        # 1. Retrieve relevant chunks
        results = vs.retrieve(user_msg, k=4)

        # 2. Build debug info
        debug_info = '\n\n---\n### 🔍 Sources Used\n'
        for i, (doc, score) in enumerate(results):
            rel = max(0.0, float(score))   # FAISS inner-product: higher = better
            src = Path(doc.metadata.get('source', '')).name
            debug_info += f'**Chunk {i+1}** — {src} (score: {rel:.3f})\n'

        # 3. Generate answer via Groq
        answer = llm.generate(user_msg, results, history)
        full_response = answer + debug_info

        history.append({'role': 'user',      'content': user_msg})
        history.append({'role': 'assistant', 'content': full_response})
        return history, ''

    except Exception as e:
        error_msg = f'❌ Error: {str(e)}'
        history.append({'role': 'user',      'content': user_msg})
        history.append({'role': 'assistant', 'content': error_msg})
        return history, ''


def do_clear_chat():
    return [], ''


def get_kb_status():
    if not vs.ready:
        return '❌  No documents loaded'
    names = [Path(p).name for p in indexed_files]
    return f'✅  {vs.count} chunks  |  {len(indexed_files)} file(s): {", ".join(names)}'


print('✅ Callbacks ready')

✅ Callbacks ready


## Step 9 — Launch Gradio App

In [ ]:
import gradio as gr

CSS = """
.app-header { text-align: center; padding: 20px 0 10px; }
.app-header h1 { font-size: 2em; margin-bottom: 4px; }
.app-header p  { color: #888; font-size: 1.1em; }
"""

# ── BUG FIX: css moved to launch() for Gradio 6 compatibility ────────────────
with gr.Blocks(title='RAG FAQ Chatbot — Groq', css=CSS) as demo:

    gr.HTML("""
    <div class='app-header'>
      <h1>🤖 Customer FAQ Chatbot</h1>
      <p>RAG · FAISS · Groq LLaMA 3.3 70B</p>
    </div>
    """)

    page = gr.Radio(
        ['💬 Chat', '📂 Upload Documents', 'ℹ️ Architecture'],
        value='💬 Chat',
        label='Navigation'
    )

    # ── CHAT PAGE ─────────────────────────────────────────────────────────────
    with gr.Column(visible=True) as chat_page:
        gr.Markdown('### 💬 Chat with your documents')

        kb_status = gr.Textbox(
            value='❌  No documents loaded',
            label='Knowledge Base Status',
            interactive=False
        )
        refresh_btn = gr.Button('🔄 Refresh KB Status', size='sm')

        chatbox = gr.Chatbot(
            height=420,
            label='Conversation',
            type='messages'   # use messages format (dict-based)
        )

        with gr.Row():
            msg_input = gr.Textbox(
                placeholder='Ask a question about your documents...',
                label='Your message',
                scale=8,
                show_label=False
            )
            send_btn = gr.Button('Send ➤', variant='primary', scale=1)

        clear_chat_btn = gr.Button('🗑️ Clear Chat', size='sm')

        gr.Examples(
            examples=[
                ['What is your return policy?'],
                ['How do I reset my password?'],
                ['What are your business hours?'],
                ['How can I track my order?'],
            ],
            inputs=msg_input,
            label='💡 Example Questions'
        )

        refresh_btn.click(fn=get_kb_status, outputs=kb_status)
        send_btn.click(fn=do_chat, inputs=[msg_input, chatbox], outputs=[chatbox, msg_input])
        msg_input.submit(fn=do_chat, inputs=[msg_input, chatbox], outputs=[chatbox, msg_input])
        clear_chat_btn.click(fn=do_clear_chat, outputs=[chatbox, msg_input])

    # ── UPLOAD PAGE ───────────────────────────────────────────────────────────
    with gr.Column(visible=False) as upload_page:
        gr.Markdown('### 📂 Upload & Index Documents')
        gr.Markdown('Supported formats: **PDF**, **DOCX**, **TXT**')

        file_input = gr.File(
            file_count='multiple',
            file_types=['.pdf', '.docx', '.txt'],
            label='Select files'
        )

        with gr.Row():
            upload_btn   = gr.Button('📥 Index Documents', variant='primary')
            clear_kb_btn = gr.Button('🗑️ Clear Knowledge Base')

        upload_out = gr.Textbox(lines=8, label='Indexing Log', interactive=False)

        upload_btn.click(fn=do_upload,   inputs=file_input, outputs=upload_out)
        clear_kb_btn.click(fn=do_clear_kb, outputs=upload_out)

    # ── ARCHITECTURE PAGE ─────────────────────────────────────────────────────
    with gr.Column(visible=False) as arch_page:
        gr.Markdown("""
### ℹ️ System Architecture

| Component | Technology |
|-----------|------------|
| Embeddings | `sentence-transformers/all-MiniLM-L6-v2` |
| Vector DB | FAISS (CPU) |
| LLM Provider | **Groq** |
| LLM Model | Groq Model |
| UI | Gradio Blocks |
| Chunking | LangChain RecursiveCharacterTextSplitter |
        """
        )

    # ── Page switching ────────────────────────────────────────────────────────
    def switch_page(p):
        return (
            gr.update(visible=(p == '💬 Chat')),
            gr.update(visible=(p == '📂 Upload Documents')),
            gr.update(visible=(p == 'ℹ️ Architecture')),
        )

    page.change(
        fn=switch_page,
        inputs=page,
        outputs=[chat_page, upload_page, arch_page]
    )

demo.queue()
demo.launch(
    share=True,
    server_name='0.0.0.0',
    show_error=True,
    debug=True
)


/tmp/ipykernel_5304/121089043.py:10: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title='RAG FAQ Chatbot — Groq', css=CSS) as demo:
/tmp/ipykernel_5304/121089043.py:36: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbox = gr.Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://964c658a4504270cbb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
